In [2]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('/kaggle/input/datasets/maneethr/happieness/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())


Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [3]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

In [7]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # ascending -> happiest ends up on top in a horizontal bar

print(region_avg)

# Step 2: Build your chart
global_avg = df['Happiness_Score'].mean()
top_region, top_val = region_avg.iloc[-1]['Region'], region_avg.iloc[-1]['Happiness_Score']
low_region, low_val = region_avg.iloc[0]['Region'], region_avg.iloc[0]['Happiness_Score']

HIGHLIGHT, MUTED = '#1B7F79', '#C9CED6'
colors = [HIGHLIGHT if r == top_region else MUTED for r in region_avg['Region']]

fig1 = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker_color=colors,
    text=region_avg['Happiness_Score'].round(2),
    textposition='outside',
    cliponaxis=False,
    hovertemplate='%{y}: %{x:.2f}<extra></extra>',
))

# Global average reference line
fig1.add_vline(x=global_avg, line_dash='dash', line_color='#444', line_width=1.5,
               annotation_text=f'Global avg {global_avg:.2f}',
               annotation_position='bottom right', annotation_font_size=12)

fig1.update_layout(
    title=dict(
        text=(f'<b>{top_region} is the happiest region ({top_val:.1f}), '
              f'{top_val - low_val:.1f} points above {low_region}</b>'
              f'<br><sup>Average World Happiness Report 2023 score by region (0-10 scale)</sup>'),
        x=0.01, xanchor='left'),
    xaxis=dict(range=[0, region_avg['Happiness_Score'].max() * 1.12],  # zero baseline
               title='Average happiness score', showgrid=True, gridcolor='#EEE', zeroline=False),
    yaxis=dict(title='', automargin=True),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=14, color='#222'),
    showlegend=False,
    margin=dict(l=20, r=40, t=100, b=50),
    height=500, width=900,
)
fig1.show()

                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


In [5]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build your chart
top_mean, bottom_mean = top8['Happiness_Score'].mean(), bottom8['Happiness_Score'].mean()
gap = top_mean - bottom_mean

GROUP_COLORS = {'Top 8': '#1B7F79', 'Bottom 8': '#D1603D'}

fig2 = go.Figure()
for group in ['Bottom 8', 'Top 8']:
    sub = combined[combined['Group'] == group]
    fig2.add_trace(go.Bar(
        x=sub['Happiness_Score'], y=sub['Country'], orientation='h',
        name=group, marker_color=GROUP_COLORS[group],
        text=sub['Happiness_Score'].round(2), textposition='outside', cliponaxis=False,
        hovertemplate='%{y}: %{x:.2f}<extra></extra>',
    ))

# Keep the sorted order: lowest at bottom, highest at top
fig2.update_yaxes(categoryorder='array', categoryarray=combined['Country'].tolist(), title='')

# Visual separator between the two groups (between bar index 7 and 8)
fig2.add_hline(y=7.5, line_dash='dot', line_color='#888', line_width=1.5)

# Gap annotation
fig2.add_annotation(
    x=combined['Happiness_Score'].max() * 1.0, y=7.5, xref='x', yref='y',
    text=f'<b>{gap:.1f}-point gap</b><br>between group averages<br>({top_mean:.1f} vs {bottom_mean:.1f})',
    showarrow=False, xanchor='right', yanchor='middle',
    bgcolor='white', bordercolor='#888', borderwidth=1, font=dict(size=13),
)

# Stretch goal: global average reference line
fig2.add_vline(x=global_avg, line_dash='dash', line_color='#444', line_width=1.5,
               annotation_text=f'Global avg {global_avg:.2f}', annotation_position='top',
               annotation_font_size=12)

fig2.update_layout(
    title=dict(
        text=(f'<b>The happiest countries score {top_mean / bottom_mean:.1f}x higher than the least happy</b>'
              f'<br><sup>Top 8 vs bottom 8 countries, World Happiness Report 2023</sup>'),
        x=0.01, xanchor='left'),
    xaxis=dict(range=[0, combined['Happiness_Score'].max() * 1.12],  # zero baseline
               title='Happiness score', showgrid=True, gridcolor='#EEE', zeroline=False),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=14, color='#222'),
    legend=dict(orientation='h', y=-0.12, x=0),
    margin=dict(l=20, r=40, t=100, b=70),
    bargap=0.25, height=700, width=900,
)
fig2.show()

Global average: 5.81


In [6]:
# Stretch goal — grouped bar chart
# --------------------------------
# Note: after renaming the columns in the setup cell, the GDP column is called 'GDP'.
regions = ['Western Europe', 'Latin America', 'East Asia', 'Sub-Saharan Africa', 'South Asia']

missing = [r for r in regions if r not in df['Region'].unique()]
if missing:
    print("Warning: these regions are not spelled this way in the data:", missing)
    print("Available regions:", sorted(df['Region'].unique()))

factors = (df[df['Region'].isin(regions)]
           .groupby('Region')[['GDP', 'Freedom']].mean()
           .reindex([r for r in regions if r in df['Region'].unique()])
           .sort_values('GDP', ascending=False))
print(factors)

fig3 = go.Figure()
fig3.add_trace(go.Bar(x=factors.index, y=factors['GDP'], name='GDP per capita',
                      marker_color='#1B7F79', text=factors['GDP'].round(2), textposition='outside'))
fig3.add_trace(go.Bar(x=factors.index, y=factors['Freedom'], name='Freedom',
                      marker_color='#E3A72F', text=factors['Freedom'].round(2), textposition='outside'))

top_gdp = factors['GDP'].idxmax()
fig3.update_layout(
    barmode='group',
    title=dict(
        text=(f'<b>Economic strength and freedom do not always travel together</b>'
              f'<br><sup>{top_gdp} leads on GDP; average GDP and Freedom sub-scores by region, 2023</sup>'),
        x=0.01, xanchor='left'),
    yaxis=dict(range=[0, max(factors['GDP'].max(), factors['Freedom'].max()) * 1.2],  # zero baseline
               title='Average sub-score', showgrid=True, gridcolor='#EEE', zeroline=False),
    xaxis=dict(title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=14, color='#222'),
    legend=dict(orientation='h', y=-0.15, x=0),
    margin=dict(l=20, r=40, t=100, b=70),
    height=500, width=950,
)
fig3.show()

Available regions: ['Central and Eastern Europe', 'East Asia', 'Latin America and Caribbean', 'Middle East and North Africa', 'North America and ANZ', 'South Asia', 'Southeast Asia', 'Sub-Saharan Africa', 'Western Europe']
                          GDP   Freedom
Region                                 
Western Europe      10.963533  0.896067
East Asia           10.312000  0.754000
South Asia           7.152250  0.546250
Sub-Saharan Africa   7.082000  0.599429
